# Utgiftredovisningsanalys

Detta anteckningsblock visar hur man skapar agenter som använder plugins för att bearbeta reseutgifter från lokala kvittobilder, generera ett utgiftsredovisningsmail och visualisera utgiftsdata med ett cirkeldiagram. Agenter väljer dynamiskt funktioner baserat på uppgiftskontexten.

Steg:
1. OCR-agenten bearbetar den lokala kvittobilden och extraherar resekostnadsdata.
2. E-postagenten genererar ett utgiftsredovisningsmail.

### Exempel på ett reseutgiftsscenario:
Föreställ dig att du är en anställd som reser för ett affärsmöte i en annan stad. Ditt företag har en policy att ersätta alla rimliga reserelaterade kostnader. Här är en sammanställning av potentiella resekostnader:
- Transport:
Flygresa tur och retur från din hemstad till destinationsstaden.
Taxi eller samåkningsjänster till och från flygplatsen.
Lokal transport i destinationsstaden (som kollektivtrafik, hyrbilar eller taxibilar).

- Boende:
Hotellvistelse i tre nätter på ett mellanklassaffärshotell nära mötesplatsen.

- Måltider:
Daglig måltidsersättning för frukost, lunch och middag, baserat på företagets traktamentspolicy.

- Diverse kostnader:
Parkeringsavgifter vid flygplatsen.
Internetavgifter på hotellet.
Dricks eller små serviceavgifter.

- Dokumentation:
Du skickar in alla kvitton (flyg, taxi, hotell, måltider etc.) och en ifylld utgiftsredovisning för ersättning.


## Importera nödvändiga bibliotek

Importera de nödvändiga biblioteken och modulerna för anteckningsboken.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## Definiera Kostnadsmodeller

 Skapa en Pydantic-modell för enskilda kostnader och en ExpenseFormatter-klass för att omvandla en användarfråga till strukturerad kostnadsdata.

 Varje kostnad kommer att representeras i formatet:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Definiera verktyg - Generera e-post

Skapa en verktygsfunktion för att generera ett e-postmeddelande för att skicka in en utgiftsansökan.
- Det här verktyget använder `@tool`-dekorationen från Microsoft Agent Framework.
- Det beräknar det totala beloppet för utgifterna och formaterar detaljerna till ett e-postmeddelande.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Verktyg för att extrahera resekostnader från kvittobilder

Skapa en verktygsfunktion för att extrahera resekostnader från kvittobilder.
- Detta verktyg använder `@tool`-dekoreraren från Microsoft Agent Framework.
- Det läser kvittobilden, kodar den som base64 och returnerar data-URI:n för att agenten ska analysera.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Behandling av utgifter

Definiera agenterna och koppla ihop dem i ett sekventiellt arbetsflöde med `WorkflowBuilder`.
- OCR-agenten extraherar strukturerad utgiftsdata från kvittobilden med hjälp av verktyget `load_receipt_image`.
- Email-agenten tar den extraherade datan och genererar ett professionellt e-postmeddelande för utgiftsrapport med hjälp av verktyget `generate_expense_email`.
- `WorkflowBuilder` med `add_edge` skapar en sekventiell kedja: OCR-agent → Email-agent.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Main function

Bygg den sekventiella arbetsflödet och kör det för att bearbeta kvittobilden och generera e-postmeddelandet för kostnadsersättningen.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, bör du vara medveten om att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål ska betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för eventuella missförstånd eller feltolkningar som uppstår vid användning av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
